# Archive CityBikes — plan, schéma et transformation

Ce notebook permet de vérifier les fichiers mensuels disponibles avant téléchargement, d'inspecter le schéma réel d'un Parquet puis de produire la Silver horaire. La collecte directe reste séparée dans `02_citybikes_stream.ipynb`.

Pour une première lecture légère : `python scripts/citybikes_archive.py --years 2025 --networks vlille`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config' / 'cities.yml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.citybikes_archive import build_plan, build_session, configured_networks, transform

ANNEES = [2025]
RESEAUX = ['vlille']
ARCHIVE_ROOT = PROJECT_ROOT / 'data' / 'citybikes-archive'
session = build_session()
plan = build_plan(session, ANNEES, RESEAUX)
print(f'{len(plan)} fichier(s), {sum(item["size"] for item in plan) / 1e6:.1f} Mo')
display(pd.DataFrame(plan))

## Télécharger et inspecter

La cellule suivante est volontairement désactivée : le téléchargement complet est lancé une seule fois par le membre qui possède la machine de collecte.

In [ ]:
TELECHARGER = False
if TELECHARGER:
    from concurrent.futures import ThreadPoolExecutor
    from scripts.citybikes_archive import download_one
    ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=2) as pool:
        fichiers = list(pool.map(lambda item: download_one(session, ARCHIVE_ROOT, item), plan))
    print(f'{len(fichiers)} fichier(s) téléchargé(s)')
else:
    print('Téléchargement désactivé : TELECHARGER=False')

In [ ]:
parquets = sorted(ARCHIVE_ROOT.glob('*/*.parquet'))
if not parquets:
    print('Aucun Parquet local. Activez TELECHARGER ou lancez le script archive.')
else:
    sample = parquets[0]
    print('Échantillon :', sample)
    sample_df = pd.read_parquet(sample)
    print('Colonnes :', list(sample_df.columns))
    display(sample_df.head())
    display(sample_df.dtypes.rename('type').to_frame())

## Transformer en Silver horaire

Après inspection, la transformation mappe les colonnes réelles vers le contrat du flux direct : `city_id`, `network_id`, `station_id`, `station_name`, coordonnées, `bikes_available`, `free_slots`, `observed_at_utc` et `hour_utc`.

In [ ]:
if parquets:
    silver_paths = transform(parquets, ARCHIVE_ROOT)
    for path in silver_paths:
        result = pd.read_parquet(path)
        print(path, '—', len(result), 'lignes')
        display(result.head())